# MMCultureQA – Task 2 (Textual VQA): QLoRA-Finetuning von Qwen2.5-Omni-3B

Läuft komplett auf einer **Colab T4** (16 GB). Vorher: **Laufzeit → Laufzeittyp ändern → T4-GPU**.

Ablauf:

1. **Install**: Pakete installieren (danach startet die Laufzeit einmal neu, dann nochmal *Alle ausführen*).
2. **Config und Google Drive**: Checkpoints, Vorhersagen und Scores landen auf Drive und überleben so einen Abbruch.
3. **Daten**: `train` (zum Trainieren) und `dev` (zum Auswerten) laden, dazu die Bilder. Außerdem wird geprüft, ob Bilder in beiden Splits vorkommen.
4. **Baseline v1**: Die alten Baseline-Antworten (`results/prediction_qa_en_full.jsonl`) werden bereinigt und neu bewertet. Dafür braucht es keine GPU.
5. **Modell und LoRA** laden.
6. **Training** auf `train`, mit Checkpoints auf Drive. Nach einem Abbruch einfach neu ausführen, dann geht es weiter.
7. **Auswertung** auf `dev` mit dem offiziellen Scorer (BERTScore-F1, mmBERT Layer 17).
8. **devtest**: Vorhersagen für das EvalAI-Leaderboard erzeugen.

Vergleichswert: Die Baseline v0 erreicht **BERTScore-F1 = 0.7392** auf `dev` (`results/scores_v0.json`).

**Wichtig:** Trainiert wird nur auf `train`, ausgewertet nur auf `dev`. Das Modell sieht `dev` beim Training also nie, und die Auswertung bleibt aussagekräftig.

## 1. Install

In [ ]:
# Einmal installieren, dann startet die Laufzeit neu. Danach nochmal "Alle ausführen".
import os
_FLAG = "/content/.mmcqa_ft_deps"
if not os.path.exists(_FLAG):
    !pip install -q -U "transformers>=4.52.0,<5" "peft>=0.13" accelerate bitsandbytes qwen-omni-utils torchvision soundfile librosa datasets "huggingface_hub[hf_transfer]" pyarrow
    !pip install -q bert_score==0.3.12 sacrebleu==2.5.1 rouge-score sentencepiece
    open(_FLAG, "w").close()
    print("Pakete installiert, Laufzeit startet neu. Danach nochmal 'Alle ausführen'.")
    import IPython; IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Pakete bereit")

## 2. Config und Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

TASK, TRACK, REGION = "qa", "en", "mena"
REPO   = "QCRI/MMCQA-SemEval27"
CONFIG = f"{TASK}_{REGION}_{TRACK}"

MODEL      = "Qwen/Qwen2.5-Omni-3B"
MAX_PIXELS = 512 * 28 * 28     # gleich wie die Baseline; 256*28*28 halbiert die Bild-Tokens (schneller, etwas schlechter)
ROOT       = "/content/mmcqa"  # Arbeitskopie der Bilder (lokal, nicht auf Drive: viel schneller beim Training)
DRIVE_DATA = "/content/drive/MyDrive/mmcqa"   # dein Datenordner auf Drive (images/… und/oder die ZIP-Archive)

# Erst ein kurzer Probelauf (≈ 10 min), um Fehler früh zu finden. Läuft er durch: auf False setzen und neu starten.
SMOKE_TEST = True

# --- Training ---
N_TRAIN     = 2000      # erst mal ein Teil von train (max. 10.000). 2000 Beispiele ≈ 1 h auf der T4
N_VAL       = 200       # aus train abgezweigt, nur für den Validierungs-Loss (dev bleibt unberührt)
EPOCHS      = 1
LR          = 1e-4
GRAD_ACCUM  = 8         # effektive Batchgröße = 8 (Batch 1 × 8)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MAX_LEN     = 1280      # längere Beispiele werden übersprungen
SAVE_EVERY  = 50        # Optimizer-Schritte zwischen Checkpoints
EVAL_EVERY  = 50
SEED        = 42

# --- Auswertung ---
EVAL_LIMIT  = None      # None = alle 1000 dev-Fragen (≈ 25–35 min); zum Testen z. B. 100
MAX_NEW     = 64

if SMOKE_TEST:
    N_TRAIN, N_VAL, EVAL_LIMIT, SAVE_EVERY, EVAL_EVERY = 48, 8, 20, 3, 3

RUN_NAME = f"qlora_r{LORA_R}_n{N_TRAIN}" + ("_smoketest" if SMOKE_TEST else "")
DRIVE    = f"/content/drive/MyDrive/mmcqa_finetune/{RUN_NAME}"
CKPT     = f"{DRIVE}/checkpoint"     # LoRA-Adapter + Optimizer-Zustand
os.makedirs(DRIVE, exist_ok=True)
print(CONFIG, "->", DRIVE)

## 3. Daten

`train` hat 10.000 Fragen mit Antworten, `dev` hat 1.000.

Die Bilder werden in dieser Reihenfolge gesucht:
1. in deinem Drive-Ordner `MyDrive/mmcqa/images/`, von dort nach `/content/mmcqa` kopiert
2. in einem ZIP-Archiv in `MyDrive/mmcqa/` (`images_mena_<split>.zip` oder `archives/images_mena_<split>.zip`)
3. sonst Download von Hugging Face

Aus den ZIPs werden nur die Bilder entpackt, die wirklich gebraucht werden.

In [ ]:
import random, zipfile
from datasets import load_dataset
from huggingface_hub import hf_hub_download

train_all = load_dataset(REPO, CONFIG, split="train").to_list()
dev_rows  = load_dataset(REPO, CONFIG, split="dev").to_list()

rng = random.Random(SEED)
rng.shuffle(train_all)
val_rows   = train_all[:N_VAL]
train_rows = train_all[N_VAL:N_VAL + N_TRAIN]
print(f"train: {len(train_rows)} | val (aus train): {len(val_rows)} | dev: {len(dev_rows)}")

import shutil

def unpack(split, rows):
    wanted = {r["image"] for r in rows}
    todo = {m for m in wanted if not os.path.isfile(os.path.join(ROOT, m))}
    # 1) einzelne Bilder aus dem Drive-Ordner kopieren
    n_drive = 0
    for m in sorted(todo):
        src = os.path.join(DRIVE_DATA, m)
        if os.path.isfile(src):
            os.makedirs(os.path.dirname(os.path.join(ROOT, m)), exist_ok=True)
            shutil.copy(src, os.path.join(ROOT, m))
            n_drive += 1
    todo = {m for m in todo if not os.path.isfile(os.path.join(ROOT, m))}
    n_zip, source = 0, "-"
    if todo:
        # 2) ZIP auf Drive, 3) sonst ZIP von Hugging Face
        name = f"images_{REGION}_{split}.zip"
        local = [p for p in (f"{DRIVE_DATA}/{name}", f"{DRIVE_DATA}/archives/{name}") if os.path.isfile(p)]
        path = local[0] if local else hf_hub_download(REPO, f"archives/{name}", repo_type="dataset")
        source = "Drive" if local else "Hugging Face"
        with zipfile.ZipFile(path) as zf:
            for m in zf.namelist():
                if m in todo:
                    zf.extract(m, ROOT)
                    n_zip += 1
    print(f"{split}: {len(wanted)} Bilder gebraucht | {n_drive} aus Drive-Ordner kopiert | "
          f"{n_zip} aus ZIP ({source}) | Rest war schon in {ROOT}")

unpack("train", train_rows + val_rows)
unpack("dev", dev_rows)
missing = [r for r in train_rows + val_rows + dev_rows if not os.path.isfile(os.path.join(ROOT, r["image"]))]
assert not missing, f"{len(missing)} Bilder fehlen, z. B. {missing[0]['image']}"

# Überlappung prüfen: kommen dieselben Bilder/Fragen in train und dev vor?
all_train_ids  = {r["id"] for r in train_all}
all_train_imgs = {r["image"] for r in train_all}
all_train_q    = {(r["question"] or "").strip().lower() for r in train_all}
print("dev-IDs auch in train:      ", sum(r["id"] in all_train_ids for r in dev_rows))
print("dev-Bilder auch in train:   ", sum(r["image"] in all_train_imgs for r in dev_rows))
print("dev-Fragen wörtlich in train:", sum((r["question"] or "").strip().lower() in all_train_q for r in dev_rows))

import statistics
print("Länge der Referenzantworten (Wörter): Median",
      statistics.median(len(r["answer"].split()) for r in train_all))
train_rows[0]

## 4. Antworten bereinigen und Baseline v1

Die Baseline v0 redet oft weiter: Sie stellt Rückfragen („So, what do you think…?“) und erfindet ganze neue Dialogrunden („Human: …“). Das drückt die BERTScore-**Precision** (0,69 gegenüber 0,80 Recall).

`clean_answer` schneidet deshalb Folgendes weg:
- alles ab `Human:` / `User:`
- Rückfragen (Sätze, die mit `?` enden)
- einen letzten Satz, der durch `max_new_tokens` mittendrin abgeschnitten wurde
- Füllwörter am Anfang („Well, you know, …“)

Hier werden die **alten v0-Antworten** bereinigt und neu bewertet. Das läuft ohne GPU und zeigt, wie viel allein die Bereinigung bringt. Diese Zahl ist **Baseline v1**, der faire Vergleichswert fürs Finetuning.

In [ ]:
import re, json

STOP_MARKERS = ["\nHuman:", "Human:", "\nUser:", "\nAssistant:", "<|im_end|>"]
FILLER = re.compile(r"^(?:(?:well|oh|hmm+|um+|uh+|you know|so|okay|ok|ah|yeah|alright)[,!.]?\s+)+", re.I)

def clean_answer(text: str) -> str:
    for m in STOP_MARKERS:                       # 1) alles ab einer erfundenen Dialogrunde weg
        i = text.find(m)
        if i != -1:
            text = text[:i]
    text = " ".join(text.split())
    sents = re.split(r"(?<=[.!?])\s+", text)
    if len(sents) > 1 and not re.search(r"[.!?]$", sents[-1]):
        sents = sents[:-1]                       # 2) abgeschnittener letzter Satz
    kept = [s for s in sents if not s.endswith("?")] or sents[:1]   # 3) Rückfragen
    out = FILLER.sub("", " ".join(kept).strip())  # 4) Füllwörter am Anfang
    return out[:1].upper() + out[1:] if out else text.strip()

print(clean_answer("Well, you know, it's the Burj Khalifa. So, what do you think?\nHuman: Next question"))

In [ ]:
# offizieller Scorer + dein Repo (für die v0-Vorhersagen)
!git clone -q https://github.com/mmcultureqa-semeval27/mmcultureqa.git /content/mmcultureqa 2>/dev/null || git -C /content/mmcultureqa pull -q
!git clone -q https://github.com/Nachotst/Task-6-Multilingual-and-Multimodal-Culturally-Grounded-Spoken-and-Visual-Question-Answering.git /content/project 2>/dev/null || git -C /content/project pull -q

GOLD = f"/content/gold_dev_{TRACK}.jsonl"
with open(GOLD, "w", encoding="utf-8") as fh:
    for r in dev_rows:
        fh.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

def write_preds(path, answers):
    with open(path, "w", encoding="utf-8") as fh:
        for i, a in answers.items():
            fh.write(json.dumps({"id": i, "answer": a, "lang": TRACK}, ensure_ascii=False) + "\n")

def score(pred_path, name):
    out = f"{DRIVE}/scores_{name}.json"
    !python /content/mmcultureqa/scorer/score.py --gold {GOLD} --pred {pred_path} --lang {TRACK} --out {out}
    with open(out) as fh:
        res = json.load(fh)["submission_result"]
    print(f"==> {name}: BERTScore-F1 = {res['BERTScore-F1']:.4f}")
    return res

v0 = [json.loads(l) for l in open("/content/project/results/prediction_qa_en_full.jsonl", encoding="utf-8")]
write_preds(f"{DRIVE}/prediction_v0.jsonl", {p["id"]: p["answer"] for p in v0})
write_preds(f"{DRIVE}/prediction_v1_clean.jsonl", {p["id"]: clean_answer(p["answer"]) for p in v0})
scores = {}
scores["v0 (Baseline)"]         = score(f"{DRIVE}/prediction_v0.jsonl", "v0")
scores["v1 (v0 + Bereinigung)"] = score(f"{DRIVE}/prediction_v1_clean.jsonl", "v1_clean")

## 5. Modell und LoRA

In [ ]:
import torch
from transformers import BitsAndBytesConfig, Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration
from peft import LoraConfig, get_peft_model, PeftModel

assert torch.cuda.is_available(), "Keine GPU: Laufzeit → Laufzeittyp ändern → T4-GPU"
print("GPU:", torch.cuda.get_device_name(0))
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4: float16

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL, max_pixels=MAX_PIXELS)
base = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=DTYPE, device_map="auto", quantization_config=quant, attn_implementation="sdpa")

for p in base.parameters():          # alles einfrieren, trainiert werden nur die LoRA-Gewichte
    p.requires_grad_(False)
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base.config.use_cache = False

# LoRA nur auf das Sprachmodell (model.layers.*), nicht auf den Vision- oder Audio-Encoder
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  target_modules=r"^model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$")
model = get_peft_model(base, lora)
for n, p in model.named_parameters():  # LoRA-Gewichte in float32: stabiler mit float16-Autocast
    if p.requires_grad:
        p.data = p.data.float()
n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
print("LoRA-Module:", n_lora)        # erwartet: 36 Layer × 7 = 252
assert n_lora > 0, "LoRA hat keine Module gefunden: target_modules prüfen (print(model) zeigt die Namen)"
model.print_trainable_parameters()

## 6. Prompt und Trainingsbeispiele

Der Prompt ist **beim Training und bei der Auswertung derselbe**, das ist wichtig. Er verlangt direkt eine knappe, faktische Antwort ohne Rückfragen.

Beim Training lernt das Modell nur die **Antwort** (`labels`). System-Prompt, Bild und Frage sind mit `-100` maskiert.

In [ ]:
from qwen_omni_utils import process_mm_info

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
LANGUAGE = {"en": "English", "msa": "Modern Standard Arabic", "arz": "Egyptian Arabic", "ajp": "Levantine Arabic"}
INSTRUCTION = ("Answer the question about the image in {language} with one or two complete, factual sentences. "
               "Name the specific place, object, food or tradition when you can. "
               "Do not ask questions and do not add comments.")

def build_conversation(row, answer=None):
    content = [{"type": "image", "image": os.path.join(ROOT, row["image"])},
               {"type": "text", "text": row["question"]},
               {"type": "text", "text": INSTRUCTION.format(language=LANGUAGE.get(TRACK, "the language of the question"))}]
    conv = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]
    if answer is not None:
        conv.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return conv

def encode_train(row):
    conv = build_conversation(row, row["answer"])
    prompt_text = processor.apply_chat_template(conv[:-1], add_generation_prompt=True, tokenize=False)
    full_text   = processor.apply_chat_template(conv, add_generation_prompt=False, tokenize=False)
    assert full_text.startswith(prompt_text), "Chat-Template: Prompt ist kein Präfix des vollen Texts"
    _, images, _ = process_mm_info(conv, use_audio_in_video=False)
    inputs = processor(text=full_text, images=images, return_tensors="pt", padding=False)
    answer_ids = processor.tokenizer(full_text[len(prompt_text):], add_special_tokens=False)["input_ids"]
    labels = inputs["input_ids"].clone()
    labels[:, :-len(answer_ids)] = -100          # nur die Antwort (+ <|im_end|>) zählt für den Loss
    inputs["labels"] = labels
    return inputs

ex = encode_train(train_rows[0])
print("Tokens:", ex["input_ids"].shape[1], "| davon Antwort:", int((ex["labels"] != -100).sum()))
print("Gelernter Teil:", repr(processor.tokenizer.decode(ex["input_ids"][0][ex["labels"][0] != -100])))

## 7. Training

Das Training läuft in einer eigenen Schleife: Batch 1, Gradient Accumulation, float16-Autocast und GradScaler. Alle `SAVE_EVERY` Schritte landen Adapter und Optimizer-Zustand auf Drive. **Bricht Colab ab:** Zellen 1–7 neu ausführen, dann geht es beim letzten Checkpoint weiter.

In [ ]:
import time, math
import torch.nn.functional as F
from peft import set_peft_model_state_dict, get_peft_model_state_dict
from safetensors.torch import load_file

def to_device(inputs):
    out = {}
    for k, v in inputs.items():
        v = v.to(model.device)
        out[k] = v.to(DTYPE) if v.is_floating_point() else v
    return out

def loss_on(inputs):
    inputs = to_device(inputs)
    labels = inputs.pop("labels")
    with torch.autocast("cuda", dtype=DTYPE):
        logits = model(**inputs, use_cache=False).logits
    return F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                           labels[:, 1:].reshape(-1), ignore_index=-100)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = []
    for r in val_rows:
        ex = encode_train(r)
        if ex["input_ids"].shape[1] <= MAX_LEN:
            losses.append(loss_on(ex).item())
    model.train()
    return sum(losses) / len(losses)

params = [p for p in model.parameters() if p.requires_grad]
opt    = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
total_steps  = math.ceil(len(train_rows) * EPOCHS / GRAD_ACCUM)
warmup_steps = max(1, int(0.05 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup_steps)
                                          * max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps)))

step, history = 0, []
if os.path.exists(f"{CKPT}/state.pt"):        # nach einem Abbruch weitermachen
    set_peft_model_state_dict(model, load_file(f"{CKPT}/adapter_model.safetensors"))
    st = torch.load(f"{CKPT}/state.pt", weights_only=False)
    opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
    step, history = st["step"], st["history"]
    print(f"Checkpoint gefunden: weiter ab Schritt {step}/{total_steps}")

def save_ckpt():
    model.save_pretrained(CKPT)
    torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                "step": step, "history": history}, f"{CKPT}/state.pt")

order = [r for _ in range(EPOCHS) for r in train_rows]   # feste Reihenfolge, damit Resume passt
model.train()
if step == 0 and not history:
    history.append({"step": 0, "val_loss": val_loss()})
    print(f"Val-Loss vor dem Training: {history[-1]['val_loss']:.4f}")

t0, running, skipped = time.time(), [], 0
for i in range(step * GRAD_ACCUM, len(order)):
    ex = encode_train(order[i])
    if ex["input_ids"].shape[1] > MAX_LEN:
        skipped += 1
        loss = None
    else:
        loss = loss_on(ex) / GRAD_ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * GRAD_ACCUM)
    if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(order):
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        step += 1
        if step % 10 == 0:
            print(f"Schritt {step}/{total_steps} | Train-Loss {sum(running[-80:]) / max(1, len(running[-80:])):.4f}"
                  f" | LR {sched.get_last_lr()[0]:.2e} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if step % EVAL_EVERY == 0 or step == total_steps:
            history.append({"step": step, "val_loss": val_loss(),
                            "train_loss": sum(running[-80:]) / max(1, len(running[-80:]))})
            print(f"   Val-Loss: {history[-1]['val_loss']:.4f}")
        if step % SAVE_EVERY == 0 or step == total_steps:
            save_ckpt()
            print(f"   Checkpoint gespeichert ({CKPT})")
    if i % 50 == 0:
        torch.cuda.empty_cache()

model.save_pretrained(f"{DRIVE}/adapter_final")
json.dump(history, open(f"{DRIVE}/history.json", "w"), indent=1)
print(f"Fertig. Übersprungen (zu lang): {skipped}. Adapter: {DRIVE}/adapter_final")

## 8. Auswertung auf `dev`

Zwei Läufe mit demselben neuen Prompt:
- **ohne Adapter** (`prompt_only`): zeigt, was der bessere Prompt allein bringt
- **mit Adapter** (`finetuned`): zeigt, was das Finetuning zusätzlich bringt

Die Antworten werden live in eine Datei auf Drive geschrieben. Bricht die Laufzeit ab, macht die Zelle beim nächsten Mal dort weiter.

> Nach einem Neustart ohne Training: Zellen 1–6 ausführen, dann diese Zelle. Sie lädt den Adapter von Drive.

In [ ]:
if not isinstance(model, PeftModel):     # Laufzeit neu gestartet: fertigen Adapter von Drive laden
    model = PeftModel.from_pretrained(base, f"{DRIVE}/adapter_final")
model.eval()
model.base_model.model.config.use_cache = True

@torch.no_grad()
def generate_answer(row):
    conv = build_conversation(row)
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    _, images, _ = process_mm_info(conv, use_audio_in_video=False)
    inputs = to_device(processor(text=text, images=images, return_tensors="pt", padding=True))
    gen = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW, use_cache=True,
                         stop_strings=["Human:", "\nUser:"], tokenizer=processor.tokenizer)
    out = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return out.strip()

def run_split(rows, path, use_adapter=True, limit=None):
    rows = rows[:limit] if limit else rows
    done = {}
    if os.path.exists(path):
        done = {json.loads(l)["id"]: json.loads(l)["answer"] for l in open(path, encoding="utf-8") if l.strip()}
        print(f"weiter: {len(done)} Antworten schon in {path}")
    todo, t0 = [r for r in rows if r["id"] not in done], time.time()
    with open(path, "a", encoding="utf-8") as fh:
        for i, r in enumerate(todo, 1):
            try:
                if use_adapter:
                    a = generate_answer(r)
                else:
                    with model.disable_adapter():
                        a = generate_answer(r)
            except Exception as exc:          # ein kaputtes Beispiel soll nicht den ganzen Lauf beenden
                print(f"  [{r['id'][:12]}] {type(exc).__name__}: {str(exc)[:90]}"); a = ""
            fh.write(json.dumps({"id": r["id"], "answer": a, "lang": TRACK}, ensure_ascii=False) + "\n"); fh.flush()
            if i % 25 == 0 or i == len(todo):
                rate = i / (time.time() - t0)
                print(f"  {i}/{len(todo)} | {rate:.2f}/s | noch {(len(todo) - i) / rate / 60:.0f} min", flush=True)
    raw = {json.loads(l)["id"]: json.loads(l)["answer"] for l in open(path, encoding="utf-8") if l.strip()}
    return {r["id"]: raw.get(r["id"], "") for r in rows}

eval_rows = dev_rows[:EVAL_LIMIT] if EVAL_LIMIT else dev_rows
if EVAL_LIMIT:                       # beim Teil-Lauf auch die Gold-Datei passend kürzen
    with open(GOLD, "w", encoding="utf-8") as fh:
        for r in eval_rows:
            fh.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

In [ ]:
# a) nur neuer Prompt, ohne Finetuning
raw = run_split(eval_rows, f"{DRIVE}/raw_dev_prompt_only.jsonl", use_adapter=False)
write_preds(f"{DRIVE}/prediction_dev_prompt_only.jsonl", {i: clean_answer(a) for i, a in raw.items()})
scores["v2 (neuer Prompt + Bereinigung)"] = score(f"{DRIVE}/prediction_dev_prompt_only.jsonl", "v2_prompt_only")

In [ ]:
# b) mit LoRA-Adapter
raw = run_split(eval_rows, f"{DRIVE}/raw_dev_finetuned.jsonl", use_adapter=True)
write_preds(f"{DRIVE}/prediction_dev_finetuned.jsonl", {i: clean_answer(a) for i, a in raw.items()})
scores["v3 (Finetuning)"] = score(f"{DRIVE}/prediction_dev_finetuned.jsonl", "v3_finetuned")

In [ ]:
# Vergleich (achtung: v0/v1 sind auf allen 1000 dev-Fragen, v2/v3 auf EVAL_LIMIT Fragen)
print(f"{'Variante':38s} {'F1':>7s} {'P':>7s} {'R':>7s} {'ROUGE-L':>8s}")
for name, s in scores.items():
    print(f"{name:38s} {s['BERTScore-F1']:7.4f} {s['BERTScore-P']:7.4f} {s['BERTScore-R']:7.4f} {s['ROUGE-L']:8.4f}")
json.dump(scores, open(f"{DRIVE}/scores_summary.json", "w"), indent=1)

dev_by_id = {r["id"]: r for r in eval_rows}
for p in list(raw.items())[:5]:
    print("\nQ   :", dev_by_id[p[0]]["question"])
    print("Modell:", clean_answer(p[1]))
    print("Gold:", dev_by_id[p[0]]["answer"])

## 9. devtest für das Leaderboard

`devtest` hat keine Antworten. Bewertet wird es auf EvalAI: `prediction_qa_en.zip` hochladen und im Formular **Language = en** wählen.

In [ ]:
devtest_rows = load_dataset(REPO, CONFIG, split="devtest").to_list()
if SMOKE_TEST:
    devtest_rows = devtest_rows[:20]     # Probelauf: nur testen, ob Export und Format-Check gehen
unpack("devtest", devtest_rows)
raw = run_split(devtest_rows, f"{DRIVE}/raw_devtest_finetuned.jsonl", use_adapter=True)
OUT = f"{DRIVE}/prediction_{TASK}_{TRACK}.jsonl"
write_preds(OUT, {i: clean_answer(a) for i, a in raw.items()})

with open(f"/content/ids_devtest.jsonl", "w") as fh:
    for r in devtest_rows:
        fh.write(json.dumps({"id": r["id"]}) + "\n")
!python /content/mmcultureqa/format_checker/check_format.py --pred {OUT} --lang {TRACK} --ids /content/ids_devtest.jsonl

import shutil
shutil.copy(OUT, "/content/prediction.jsonl")
with zipfile.ZipFile(OUT.replace(".jsonl", ".zip"), "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("/content/prediction.jsonl", "prediction.jsonl")
print("Zum Hochladen:", OUT.replace(".jsonl", ".zip"))